In [1]:
import os
import pandas as pd
import numpy as np
import re
import geopandas as gpd

In [2]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

centroids = world.to_crs(epsg=3857).centroid

centroids_geo = centroids.to_crs(epsg=4326)

centroid_list = pd.concat([world.name, centroids_geo.x, centroids_geo.y], axis=1)
centroid_list.rename(columns={'name':'Country',0:'Longitude',1:'Latitude'},inplace=True)
centroid_list.loc[centroid_list.Country == "United Kingdom", "Country"] = "UK"

/var/folders/zg/x1cn52xj1c98v_k_hzsy2z7h0000gn/T/ipykernel_5386/2216356117.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [3]:
data_folder = os.path.join("data")
folders = [folder for folder in os.listdir(data_folder) if os.path.isdir(os.path.join(data_folder,folder)) and folder[0] != '.']

In [4]:
def replace_bracket_content(text):
    return re.sub(r'\(.*?\)', '', text)

def preprocess_df(df_path, grade):
    df = pd.read_excel(df_path, header=3)
    df.rename(columns={'Unnamed: 0':'Country', 'Unnamed: 1':'Type'}, inplace=True)
    first_nan_index = df[df.iloc[:, 0].isna()].index[0]
    df = df.loc[:first_nan_index-1]
    df['Country'] = df['Country'].apply(replace_bracket_content)
    df['Grade'] = grade
    df['Country'] = df['Country'].str.strip()
    df['stat_type'] = 'Awarded' if 'awarded' in df_path else 'Students'
    if 'First' in df_path:
        df['stat_type'] = 'First Year Students' 
    return df

In [5]:
master_df = pd.DataFrame()
for folder in folders:
    files = [file for file in os.listdir(os.path.join(data_folder,folder)) if os.path.isfile(os.path.join(data_folder,folder,file)) and 'xls' in file]
    for file in files:
        df = preprocess_df(os.path.join(data_folder,folder,file), folder.split("-")[-1])
        master_df = pd.concat([master_df,df])

In [6]:
master_df.head()

,Country,Type,population,total_10,female_10,ratio_10,female_p_10,total_11,female_11,ratio_11,...,total_20,female_20,ratio_20,female_p_20,total_21,female_21,ratio_21,female_p_21,Grade,stat_type
0,Austria,(RU),8504850.0,561,84,65.962363,0.149733,509,65,59.848204,...,825,156,97.003474,0.189091,856,163,100.648454,0.190421,bachelor,Awarded
1,Belgium,(RU),11267910.0,245,16,21.743163,0.065306,258,13,22.896881,...,405,35,35.942779,0.08642,tbp,tbp,tbp,tbp,bachelor,Awarded
2,Bulgaria,(RU),7050034.0,926,385,131.346884,0.415767,1053,427,149.360982,...,1318,422,186.949453,0.320182,1512,547,214.467051,0.361772,bachelor,Awarded
3,Czechia,(RU),10649800.0,3447,350,323.66805,0.101538,3241,338,304.324964,...,2534,429,237.938741,0.169298,tbp,tbp,tbp,tbp,bachelor,Awarded
4,Denmark,(RU),5627235.0,273,13,48.514057,0.047619,327,30,58.110244,...,659,121,117.109024,0.183612,740,154,131.503305,0.208108,bachelor,Awarded


In [7]:
master_df

,Country,Type,population,total_10,female_10,ratio_10,female_p_10,total_11,female_11,ratio_11,...,total_20,female_20,ratio_20,female_p_20,total_21,female_21,ratio_21,female_p_21,Grade,stat_type
0,Austria,(RU),8504850.0,561,84,65.962363,0.149733,509,65,59.848204,...,825,156,97.003474,0.189091,856,163,100.648454,0.190421,bachelor,Awarded
1,Belgium,(RU),11267910.0,245,16,21.743163,0.065306,258,13,22.896881,...,405,35,35.942779,0.08642,tbp,tbp,tbp,tbp,bachelor,Awarded
2,Bulgaria,(RU),7050034.0,926,385,131.346884,0.415767,1053,427,149.360982,...,1318,422,186.949453,0.320182,1512,547,214.467051,0.361772,bachelor,Awarded
3,Czechia,(RU),10649800.0,3447,350,323.66805,0.101538,3241,338,304.324964,...,2534,429,237.938741,0.169298,tbp,tbp,tbp,tbp,bachelor,Awarded
4,Denmark,(RU),5627235.0,273,13,48.514057,0.047619,327,30,58.110244,...,659,121,117.109024,0.183612,740,154,131.503305,0.208108,bachelor,Awarded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,Romania,(RU + UAS),19870000.0,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,72,34,3.623553,0.472222,47,15,2.365375,0.319149,PhD,Awarded
19,Spain,(RU + UAS),46464053.0,508,129,10.933183,0.253937,422,92,9.08229,...,609,122,13.106907,0.200328,479,97,10.309045,0.202505,PhD,Awarded
20,Switzerland,(RU + UAS),8417700.0,125,12,14.849662,0.096,104,18,12.354919,...,116,20,13.780486,0.172414,116,20,13.780486,0.172414,PhD,Awarded
21,Turkey,(RU + UAS),83150000.0,98,34,1.178593,0.346939,109,26,1.310884,...,321,91,3.860493,0.283489,298,75,3.583885,0.251678,PhD,Awarded


In [8]:
country_e_zones = pd.read_csv(os.path.join(data_folder,"additional_info","country_economic_zones.csv"))
gdp_per_capita = pd.read_csv(os.path.join(data_folder,"additional_info","country_gdp_per_capita.csv"))
country_size = pd.read_csv(os.path.join(data_folder,"additional_info","country_size.csv"))
country_zones = pd.read_csv(os.path.join(data_folder,"additional_info","country_zones.csv"))

In [9]:
master_df = master_df.merge(country_e_zones,how='left')
master_df = master_df.merge(gdp_per_capita,how='left')
master_df = master_df.merge(country_size,how="left")
master_df = master_df.merge(centroid_list,how="left")
master_df = master_df.merge(country_zones, how='left')

In [10]:
quartiles, bins = pd.qcut(master_df['GDP'], q=4, labels=[1, 2, 3, 4], retbins=True)
print("Quartile cutoff values:", bins)

Quartile cutoff values: [ 38689.  48992.  58906.  74485. 127623.]


In [67]:
master_df

,Country,Type,population,total_10,female_10,ratio_10,female_p_10,total_11,female_11,ratio_11,...,Grade,stat_type,Economic Zone,GDP,Area (km^2),Population,Longitude,Latitude,Zone,GDP Quartile
0,Austria,(RU),8504850.0,561,84,65.962363,0.149733,509,65,59.848204,...,bachelor,Awarded,1,73751,83879,9027999,14.083917,47.624674,West,3
1,Belgium,(RU),11267910.0,245,16,21.743163,0.065306,258,13,22.896881,...,bachelor,Awarded,1,70456,30689,11763650,4.577789,50.658090,West,3
2,Bulgaria,(RU),7050034.0,926,385,131.346884,0.415767,1053,427,149.360982,...,bachelor,Awarded,3,38689,110993,6385500,25.198677,42.765344,East,1
3,Czechia,(RU),10649800.0,3447,350,323.66805,0.101538,3241,338,304.324964,...,bachelor,Awarded,2,53816,78871,10900555,15.330447,49.785022,East,2
4,Denmark,(RU),5627235.0,273,13,48.514057,0.047619,327,30,58.110244,...,bachelor,Awarded,0,76687,43094,5935619,9.871311,56.082202,North,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383,Romania,(RU + UAS),19870000.0,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,PhD,Awarded,2,47903,238398,19064409,24.937800,45.891703,East,1
384,Spain,(RU + UAS),46464053.0,508,129,10.933183,0.253937,422,92,9.08229,...,PhD,Awarded,2,52779,505990,48797875,-3.614584,40.433502,South,2
385,Switzerland,(RU + UAS),8417700.0,125,12,14.849662,0.096,104,18,12.354919,...,PhD,Awarded,0,92980,41285,8902308,8.119434,46.797496,West,4
386,Turkey,(RU + UAS),83150000.0,98,34,1.178593,0.346939,109,26,1.310884,...,PhD,Awarded,2,44151,783562,85372377,35.116065,39.111747,South,1


In [68]:
master_df['Type'] = pd.Categorical(master_df['Type'], categories=['(RU)', '(UAS)', '(RU + UAS)'], ordered=True)
master_df = master_df.sort_values(by=['Type'])

In [69]:
master_df = master_df.drop_duplicates(subset=['Country','population','total_10','female_10','total_11','total_12'], keep='first')
master_df = master_df.reset_index(drop=True)

In [70]:
master_df.loc[master_df.Country == 'Italy']

,Country,Type,population,total_10,female_10,ratio_10,female_p_10,total_11,female_11,ratio_11,...,Grade,stat_type,Economic Zone,GDP,Area (km^2),Population,Longitude,Latitude,Zone,GDP Quartile
4,Italy,(RU),60782668.0,2737,456,45.029284,0.166606,2658,398,43.729571,...,master,Awarded,1,58754,301340,58968501,12.078021,42.919447,South,2
27,Italy,(RU),60782668.0,10613,1598,174.605695,0.15057,9951,1450,163.714433,...,master,Students,1,58754,301340,58968501,12.078021,42.919447,South,2
39,Italy,(RU),60782668.0,49259,n.a.,810.411942,n.a.,47909,n.a.,788.201663,...,bachelor,Students,1,58754,301340,58968501,12.078021,42.919447,South,2
72,Italy,(RU),60782668.0,1869,425,30.748897,0.227394,1825,395,30.025006,...,PhD,Students,1,58754,301340,58968501,12.078021,42.919447,South,2
101,Italy,(RU),60782668.0,528,112,8.686687,0.212121,518,109,8.522166,...,PhD,Awarded,1,58754,301340,58968501,12.078021,42.919447,South,2
111,Italy,(RU),60782668.0,10167,1346,167.268077,0.132389,10928,1485,179.788094,...,bachelor,First Year Students,1,58754,301340,58968501,12.078021,42.919447,South,2
138,Italy,(RU),60782668.0,6315,906,103.894748,0.143468,5736,804,94.369007,...,bachelor,Awarded,1,58754,301340,58968501,12.078021,42.919447,South,2


In [71]:
available = master_df.copy()
available = available.replace('n.a.',pd.NA)
available = available.replace('tbp',pd.NA)
available.dropna(how='any',inplace=True)
available = available.loc[available.Type == '(RU)']
available[['Country','Grade','Type','stat_type']].drop_duplicates().to_excel(os.path.join("available_data.xlsx"))

In [72]:
available[['Country','Grade','Type', 'stat_type']].drop_duplicates()

,Country,Grade,Type,stat_type
0,Austria,bachelor,(RU),Awarded
1,Germany,master,(RU),Awarded
3,Ireland,master,(RU),Awarded
4,Italy,master,(RU),Awarded
5,Latvia,master,(RU),Awarded
...,...,...,...,...
139,Ireland,bachelor,(RU),Awarded
141,Germany,bachelor,(RU),Awarded
143,Switzerland,bachelor,(RU),First Year Students
144,Finland,bachelor,(RU),Awarded


In [73]:
master_df.to_csv("master_df.csv")

In [74]:
master_df.Country.unique()

array(['Austria', 'Germany', 'Greece', 'Ireland', 'Italy', 'Latvia',
       'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Spain',
       'Switzerland', 'Turkey', 'UK', 'Belgium', 'Bulgaria', 'Czechia',
       'Denmark', 'Estonia', 'Finland', 'France', 'Lithuania'],
      dtype=object)